# AgentMeter — Phase 5 Pilot (Google Colab, free T4)

Per-agent resource benchmarking of one open-source LLM in a **minimal linear**
Perceive → Reason → Decide → Act network-threat-detection pipeline.

**Tahap 1 guardrails (unchanged):**
- Pipeline is the *test subject* only — minimal, generic, linear. No governance,
  correlation, alerting, or retry/self-correction loops.
- **Sequential** execution only (never parallel — it contaminates readings).
- **Mode A**: model pulled from the HF Hub and run **in-process** on the GPU, so
  per-agent VRAM is measurable via `torch.cuda` / `pynvml`.
- All config in `config.yaml` / the pilot config. Nothing hard-coded.
- GPU required: if `torch.cuda` is unavailable the pilot **STOPS** (no CPU fallback).

### ⚠️ Quantization notice (declare in your thesis)
A 7–8B model does **not** fit in fp16 on a 15 GB T4. This pilot uses **4-bit NF4**
quantization (bitsandbytes). The VRAM / latency / token numbers are therefore for
the **quantized** model, and any model-vs-model comparison must use the **same**
quantization to stay fair. This is a methodological choice you must state.

### Cost
Colab free tier has no \$ cost, but sessions are time-limited and the GPU can be
reclaimed. Save `pilot.json` as soon as the run finishes.

## 1. Confirm you have a GPU runtime
Runtime → Change runtime type → Hardware accelerator = **T4 GPU**. Then run:

In [ ]:
!nvidia-smi -L || echo 'NO GPU: set Runtime -> Change runtime type -> T4 GPU'

## 2. Get the AgentMeter code
Clones the repo and checks out the working branch. If the repo is **private**,
paste a GitHub token when prompted (input is hidden). If it is public, just press
Enter to skip.

In [ ]:
import os, getpass, subprocess
REPO = 'https://github.com/ismazahin/AgentMeter'
BRANCH = 'claude/agentmeter-phases-0-5-3ftnmx'
gh = getpass.getpass('GitHub token (press Enter if repo is public): ').strip()
url = REPO.replace('https://', f'https://{gh}@') if gh else REPO
if not os.path.isdir('AgentMeter'):
    subprocess.run(['git', 'clone', '--branch', BRANCH, url + '.git', 'AgentMeter'], check=True)
os.chdir('AgentMeter')
subprocess.run(['git', 'checkout', BRANCH], check=True)
print('cwd:', os.getcwd())
print('branch:', subprocess.run(['git','rev-parse','--abbrev-ref','HEAD'],capture_output=True,text=True).stdout.strip())

## 3. Install dependencies
Colab already ships a CUDA build of `torch` — we do **not** reinstall it (that
would risk breaking CUDA). We add the CPU-set deps plus the GPU/HF extras and
`bitsandbytes` for 4-bit.

In [ ]:
# CPU-set deps (langgraph, pandas, numpy, scipy, pyyaml) — no torch here
!pip install -q -r requirements.txt
# GPU / HF extras (torch already present on Colab)
!pip install -q transformers accelerate huggingface_hub sentencepiece pynvml bitsandbytes
import torch; print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'No CUDA GPU — switch runtime to T4 before continuing.'

## 4. Authenticate with your Hugging Face token (secure)
Entered via `getpass` — **not** hardcoded, not printed, not saved to the notebook.
You must have accepted the model's gated licence on its Hub page first.

Alternatively use a Colab Secret named `HF_TOKEN` (🔑 panel) — the cell picks it up.

In [ ]:
import os, getpass
tok = ''
try:
    from google.colab import userdata
    tok = userdata.get('HF_TOKEN') or ''
except Exception:
    pass
if not tok:
    tok = getpass.getpass('Enter your HF_TOKEN (hidden): ').strip()
assert tok, 'HF_TOKEN is required for gated models.'
os.environ['HF_TOKEN'] = tok
print('HF_TOKEN set (', len(tok), 'chars ). Not displayed.')

## 5. Choose the model (optional)
Default is `mistralai/Mistral-7B-Instruct-v0.3` from `configs/pilot_colab_t4.yaml`.
To use Llama-3-8B instead, set `MODEL_NAME` below (accept its licence first). Both
run at **4-bit NF4** so their numbers are comparable.

In [ ]:
import yaml
CONFIG = 'configs/pilot_colab_t4.yaml'
MODEL_NAME = None  # e.g. 'meta-llama/Meta-Llama-3-8B-Instruct'; None keeps the config default
with open(CONFIG) as f: cfg = yaml.safe_load(f)
if MODEL_NAME:
    cfg['model']['name'] = MODEL_NAME
    with open(CONFIG, 'w') as f: yaml.safe_dump(cfg, f, sort_keys=False)
print('Model     :', cfg['model']['name'])
print('4-bit     :', cfg['model']['hf']['quantization']['load_in_4bit'])
print('Scenarios :', cfg['dataset']['limit'])

## 6. Run the pilot
Loads the model **once** on the GPU (4-bit), then runs the instrumented 4-agent
pipeline sequentially over the scenarios. First run also downloads the weights
(~4–5 GB in 4-bit) — that is one-time and excluded from per-agent timings.

In [ ]:
from agentmeter.pilot import run_pilot
result = run_pilot(config_path=CONFIG, n=8, proj_scenarios=1000, proj_models=2,
                   json_path='results/pilot.json')
print(result.report)

## 7. Inspect and download `pilot.json`
This is the file to send back for review.

In [ ]:
import json
with open('results/pilot.json') as f: payload = json.load(f)
print('quantization:', payload['quantization'])
print('device VRAM after load (MB):', payload.get('device_vram_after_load_mb'))
print('accuracy:', payload['accuracy'])
try:
    from google.colab import files
    files.download('results/pilot.json')
except Exception as e:
    print('Download manually from the Files panel. (', e, ')')

## 8. Done — free the GPU
Runtime → Disconnect and delete runtime, so the free GPU is released for your
next session. Send me `pilot.json` and we'll review the real per-agent timing,
VRAM, and token numbers before building Phases 6–10.